<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 24 — רגרסיה ליניארית ו־R²
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
הערכת מחיר מחשב דורשת מספר, בשונה מסיווג סוג מחשב. נתחיל בישר ונקשר לדוגמת האירוס שבמצגת.

## מטרות
- להבחין בין סיווג לרגרסיה.
- לחשב תחזית ושארית.
- לפרש R².

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## המושגים הנחוצים
רגרסיה חוזה יעד מספרי. במאפיין אחד ŷ=wx+b. fit לומד מקדמים ו־predict מפיק תחזית. השארית e=y−ŷ. כאשר שונות היעד חיובית, R²=1−SSE/SST. הוא יכול להיות שלילי ואינו אחוז דיוק. ממוצע Train הוא קו בסיס מעשי, בעוד ממוצע Test מופיע בהגדרת המדד.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## דוגמה פתורה לפני קוד
במקור w=0.416 ו־b=−0.366. עבור x=2.5 התחזית 0.674. אם האמת 0.7, השארית 0.026. חישוב זה אינו אימון חדש.

</div>

In [ ]:
w = 0.416
b = -0.366
x = 2.5
prediction = w * x + b
print(round(prediction, 3))

<div dir="rtl" lang="he" style="text-align: right;">

**פלט צפוי:** 0.674

## מדוע זה חשוב?
סכום משוקלל, שגיאה ועדכון משקולות מכינים ל־DL. רגרסיה ופרספטרון אינם מוצגים כאלגוריתמי חובה ביחידה 5.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## יישום מודרך — חלק 1
מקור הקוד: ערכה 27 הקיימת. בהמשך עוברים לתרגול עצמאי באותו נושא.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

<div dir="rtl" lang="he" style="text-align: right;">

### טעינה ובדיקת קלט
טוענים את מקור הנתונים שהופיע בתרגול. בודקים שדות וממדים לפני שימוש. כתובת Raw שונה מדף התצוגה של GitHub.

</div>

In [ ]:
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df[["flipper_length_mm", "bill_length_mm", "body_mass_g"]].isna().sum())

<div dir="rtl" lang="he" style="text-align: right;">

### תשתית הנתונים והחלוקה
קראו תחילה את היעד ואת שמות המאפיינים. החלוקה שומרת התאמה בין שורות X ל־y. הפרמטר random_state מאפשר שחזור ואינו משפר דיוק בפני עצמו.

</div>

In [ ]:
# לא משלימים יעד חסר; משתמשים בעותק עבודה
model_df = df.dropna(subset=["body_mass_g"]).copy()
feature = "flipper_length_mm"
X = model_df[[feature]].copy()
y = model_df["body_mass_g"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)
print("Eligible:", len(model_df), "Train:", len(X_train), "Test:", len(X_test))
print("Train feature range:", X_train_ready[feature].min(), X_train_ready[feature].max())

<div dir="rtl" lang="he" style="text-align: right;">

### התאמה ויישום
זהו מה לומד פרמטרים ובאילו שורות נעשה fit. ודאו שכל הכנה שנלמדת מנתונים משתמשת באימון המתאים. לאחר מכן פירשו את הפלט במילים.

</div>

In [ ]:
model = LinearRegression()
model.fit(X_train_ready, y_train)
predictions = model.predict(X_test_ready)
w = model.coef_[0]
b = model.intercept_
print("Slope (g/mm):", w, "Intercept (g):", b)
new_penguin = pd.DataFrame({feature: [200]})
new_prediction = model.predict(new_penguin)
print("New feature value (mm):", 200, "Predicted mass (g):", new_prediction[0])

<div dir="rtl" lang="he" style="text-align: right;">

### המחשה חזותית
לפני הרצה הגדירו שאלה, סוגי משתנים ויחידות. אחרי הרצה פרשו את הצירים ואת הממצא. אין להסיק סיבתיות או למחוק קצה רק מן הציור.

</div>

In [ ]:
# קטע ציור נתון; ממיינים את X כדי לצייר ישר בלי חיבור בזיגזג
line_X = X_train_ready.sort_values(feature)
line_y = model.predict(line_X)
plt.figure()
plt.scatter(X_train_ready[feature], y_train, alpha=0.35, label="Train")
plt.scatter(X_test_ready[feature], y_test, alpha=0.65, label="Test")
plt.plot(line_X[feature], line_y, color="black", label="Line fitted on Train")
plt.xlabel(feature + " (mm)")
plt.ylabel("body_mass_g (g)")
plt.title("Penguins — one-feature linear regression")
plt.legend()
plt.tight_layout()
plt.show()

<div dir="rtl" lang="he" style="text-align: right;">

### המחשה חזותית
לפני הרצה הגדירו שאלה, סוגי משתנים ויחידות. אחרי הרצה פרשו את הצירים ואת הממצא. אין להסיק סיבתיות או למחוק קצה רק מן הציור.

</div>

In [ ]:
comparison = pd.DataFrame(
    {"actual_g": y_test, "predicted_g": predictions}, index=y_test.index
)
comparison["residual_g"] = comparison["actual_g"] - comparison["predicted_g"]
display(comparison.head(8))
plt.figure()
plt.scatter(comparison["predicted_g"], comparison["residual_g"], alpha=0.6)
plt.axhline(0, color="black")
plt.xlabel("Predicted mass (g)")
plt.ylabel("Residual: actual - predicted (g)")
plt.title("Test residuals")
plt.tight_layout()
plt.show()

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
# בדיקת הדוגמה הידנית בלבד, ללא אימון או CSV
toy_actual = pd.Series([10.0, 20.0, 30.0])
toy_predicted = pd.Series([12.0, 18.0, 30.0])
toy_sse = ((toy_actual - toy_predicted) ** 2).sum()
toy_sst = ((toy_actual - toy_actual.mean()) ** 2).sum()
print("Toy SSE:", toy_sse, "SST:", toy_sst, "R²:", 1 - toy_sse / toy_sst)
print("Poor prediction R²:", r2_score(toy_actual, [40, 40, 40]))

<div dir="rtl" lang="he" style="text-align: right;">

### התאמה ויישום
זהו מה לומד פרמטרים ובאילו שורות נעשה fit. ודאו שכל הכנה שנלמדת מנתונים משתמשת באימון המתאים. לאחר מכן פירשו את הפלט במילים.

</div>

In [ ]:
# ממוצע y_test משמש רק להגדרת המדד, לא ליצירת תחזיות
sse = (comparison["residual_g"] ** 2).sum()
test_mean = y_test.mean()
sst = ((y_test - test_mean) ** 2).sum()
r2_manual = 1 - sse / sst
r2_test = r2_score(y_test, predictions)
print("SSE:", sse, "SST:", sst)
print("R² manual:", r2_manual, "R² library:", r2_test)

# קו בסיס מעשי: חיזוי ממוצע שנלמד מאימון בלבד
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_ready, y_train)
baseline_predictions = baseline.predict(X_test_ready)
baseline_r2 = r2_score(y_test, baseline_predictions)
print("Training mean baseline (g):", baseline.constant_[0, 0])
print("Baseline R² on same Test:", baseline_r2)

<div dir="rtl" lang="he" style="text-align: right;">

## עצירה לפני המעבדה
האם R² שלילי הוא בהכרח שגיאת תוכנה?

**התוצר הנדרש:** מאפיין אחד, אימון, תחזית, שאריות, R² וקו בסיס. אין צורך בגזירת פתרון הריבועים הפחותים.

</div>